<a href="https://colab.research.google.com/github/carolinampessoa/TechChallengeFase5/blob/main/TechChallengeFase5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
#Instalação de libs
!pip install openai

In [21]:
#Configurar API Key (uso de LLM da OpenAI)
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Digite sua OpenAI API Key: ")

KeyboardInterrupt: Interrupted by user

In [22]:
#Importar imagens para validação
from google.colab import files

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

print("Imagem carregada:", image_path)


KeyboardInterrupt: 

In [10]:
from openai import OpenAI
import base64
import json

client = OpenAI()

def encode_image(path):
    with open(path, "rb") as img:
        return base64.b64encode(img.read()).decode("utf-8")

base64_image = encode_image(image_path)

prompt = """
Analise o diagrama de arquitetura de software presente na imagem.

Identifique todos os componentes do sistema.
Classifique cada componente em uma das categorias:

- user
- server
- database
- api
- external_system

Responda APENAS em JSON no formato:

{
  "components": [
    {"name": "...", "type": "..."}
  ]
}
"""

response = client.responses.create(
    model="gpt-4.1",
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {
                    "type": "input_image",
                    "image_url": f"data:image/png;base64,{base64_image}"
                }
            ]
        }
    ]
)

output_text = response.output_text
print(output_text)


```json
{
  "components": [
    {"name": "Usuários SEI", "type": "user"},
    {"name": "AWS Shield", "type": "external_system"},
    {"name": "Amazon CloudFront", "type": "external_system"},
    {"name": "AWS WAF", "type": "external_system"},
    {"name": "Virtual Private Cloud", "type": "server"},
    {"name": "Application Load Balancer", "type": "server"},
    {"name": "SEI / SIP (API Server)", "type": "server"},
    {"name": "Solr", "type": "server"},
    {"name": "Amazon Elastic File System (NFS) Multi-AZ", "type": "database"},
    {"name": "Amazon RDS (Primary)", "type": "database"},
    {"name": "Amazon RDS (Secondary)", "type": "database"},
    {"name": "Amazon ElastiCache (memcached) Multi-AZ", "type": "database"},
    {"name": "AWS CloudTrail", "type": "external_system"},
    {"name": "AWS Key Management Service", "type": "external_system"},
    {"name": "AWS Backup", "type": "external_system"},
    {"name": "Amazon CloudWatch", "type": "external_system"},
    {"name": "Amazon

In [13]:
import re

# Remove markdown code block delimiters if they exist
json_string = re.search(r'```json\n([\s\S]*?)\n```', output_text)
if json_string:
    clean_output_text = json_string.group(1)
else:
    clean_output_text = output_text.strip()

data = json.loads(clean_output_text)
components = data["components"]

components

[{'name': 'Usuários SEI', 'type': 'user'},
 {'name': 'AWS Shield', 'type': 'external_system'},
 {'name': 'Amazon CloudFront', 'type': 'external_system'},
 {'name': 'AWS WAF', 'type': 'external_system'},
 {'name': 'Virtual Private Cloud', 'type': 'server'},
 {'name': 'Application Load Balancer', 'type': 'server'},
 {'name': 'SEI / SIP (API Server)', 'type': 'server'},
 {'name': 'Solr', 'type': 'server'},
 {'name': 'Amazon Elastic File System (NFS) Multi-AZ', 'type': 'database'},
 {'name': 'Amazon RDS (Primary)', 'type': 'database'},
 {'name': 'Amazon RDS (Secondary)', 'type': 'database'},
 {'name': 'Amazon ElastiCache (memcached) Multi-AZ', 'type': 'database'},
 {'name': 'AWS CloudTrail', 'type': 'external_system'},
 {'name': 'AWS Key Management Service', 'type': 'external_system'},
 {'name': 'AWS Backup', 'type': 'external_system'},
 {'name': 'Amazon CloudWatch', 'type': 'external_system'},
 {'name': 'Amazon Simple Email Service (SES)', 'type': 'external_system'}]

In [15]:
stride_map = {
    "server": ["Spoofing", "Tampering", "Denial of Service"],
    "database": ["Tampering", "Information Disclosure"],
    "api": ["Spoofing", "Repudiation"],
    "user": ["Spoofing"],
    "external_system": ["Spoofing", "Tampering"]
}

def analyze_stride(components):
    results = []

    for comp in components:
        threats = stride_map.get(comp["type"], [])
        results.append({
            "component": comp["name"],
            "type": comp["type"],
            "threats": threats
        })

    return results

stride_results = analyze_stride(components)
stride_results


[{'component': 'Usuários SEI', 'type': 'user', 'threats': ['Spoofing']},
 {'component': 'AWS Shield',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'Amazon CloudFront',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'AWS WAF',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'Virtual Private Cloud',
  'type': 'server',
  'threats': ['Spoofing', 'Tampering', 'Denial of Service']},
 {'component': 'Application Load Balancer',
  'type': 'server',
  'threats': ['Spoofing', 'Tampering', 'Denial of Service']},
 {'component': 'SEI / SIP (API Server)',
  'type': 'server',
  'threats': ['Spoofing', 'Tampering', 'Denial of Service']},
 {'component': 'Solr',
  'type': 'server',
  'threats': ['Spoofing', 'Tampering', 'Denial of Service']},
 {'component': 'Amazon Elastic File System (NFS) Multi-AZ',
  'type': 'database',
  'threats': ['Tampering', 'Information Disclosure']},
 {'component'

In [16]:
def generate_report(results):
    lines = []

    for item in results:
        lines.append(f"Componente: {item['component']}")
        lines.append(f"Tipo: {item['type']}")
        lines.append(f"Ameaças STRIDE: {', '.join(item['threats'])}")
        lines.append("-" * 50)

    return "\n".join(lines)

report = generate_report(stride_results)

print(report)


Componente: Usuários SEI
Tipo: user
Ameaças STRIDE: Spoofing
--------------------------------------------------
Componente: AWS Shield
Tipo: external_system
Ameaças STRIDE: Spoofing, Tampering
--------------------------------------------------
Componente: Amazon CloudFront
Tipo: external_system
Ameaças STRIDE: Spoofing, Tampering
--------------------------------------------------
Componente: AWS WAF
Tipo: external_system
Ameaças STRIDE: Spoofing, Tampering
--------------------------------------------------
Componente: Virtual Private Cloud
Tipo: server
Ameaças STRIDE: Spoofing, Tampering, Denial of Service
--------------------------------------------------
Componente: Application Load Balancer
Tipo: server
Ameaças STRIDE: Spoofing, Tampering, Denial of Service
--------------------------------------------------
Componente: SEI / SIP (API Server)
Tipo: server
Ameaças STRIDE: Spoofing, Tampering, Denial of Service
--------------------------------------------------
Componente: Solr
Tipo: s

In [17]:
counter_prompt = f"""
Considere as seguintes ameaças STRIDE identificadas:

{json.dumps(stride_results, indent=2)}

Sugira contramedidas de segurança para cada componente.
"""

response2 = client.responses.create(
    model="gpt-4.1",
    input=counter_prompt
)

print(response2.output_text)


Claro! Abaixo está uma lista de **contramedidas recomendadas**, organizadas por componente, levando em consideração as ameaças STRIDE identificadas. Cada ameaça recebe pelo menos uma ou mais práticas de segurança condizentes com o contexto do componente:

---

### Usuários SEI (`user`)
**Ameaças:** Spoofing  
**Contramedidas:**
- Autenticação forte (MFA/autenticação de dois fatores)
- Políticas de senha robustas
- Controle de sessão (timeout, logout automático)
- Monitoramento e alerta de acessos suspeitos

---

### AWS Shield (`external_system`)
**Ameaças:** Spoofing, Tampering  
**Contramedidas:**
- Usar certificados válidos (TLS) na comunicação
- Limitar acessos administrativos via IAM
- Monitorar logs de acesso e configuração (CloudTrail)
- Configuração automatizada para impedir alterações não autorizadas

---

### Amazon CloudFront (`external_system`)
**Ameaças:** Spoofing, Tampering  
**Contramedidas:**
- Usar HTTPS para entrega de conteúdo
- Validação de origem de requests (Sign